# Detectree2: Tree Crown Detection with Adaptive Thresholding
This notebook:
1. Runs Detectree2 inference on all orthomosaics in `input/input_om/`
2. Sweeps confidence thresholds to find the value per OM that yields crown counts closest to the global median
3. Saves the final crowns to `output/input_crowns/OM{n}.gpkg`

In [2]:
# Optional installs (run only if packages are missing)
import sys, subprocess

def ensure(pkg_args):
    if isinstance(pkg_args, str):
        pkg_args = [pkg_args]
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pkg_args])
    except Exception as e:
        print(f'Failed to install {pkg_args}: {e}')

# Try imports; if missing, attempt installs.
try:
    import rasterio
except Exception:
    ensure(['rasterio'])

try:
    import geopandas as gpd
except Exception:
    ensure(['geopandas', 'pyproj', 'shapely', 'fiona'])

# Detectron2 stack (may already be present in the environment).
try:
    import detectron2
except Exception:
    # Ensure PyTorch CPU build (safe on macOS) before detectron2
    try:
        import torch
    except Exception:
        ensure(['torch', 'torchvision', 'torchaudio'])
    # Try installing detectron2 from source; may take time.
    ensure(['git+https://github.com/facebookresearch/detectron2.git'])

try:
    import detectree2
except Exception:
    ensure(['git+https://github.com/PatBall1/detectree2.git'])

print('Environment check complete.')

  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/150.8 MB ? eta -:--:--Collecting mpmath<1.4,>=1.1.0 (from sympy->torch)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 MB 6.1 MB/s  0:00:24m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 MB 6.1 MB/s  0:00:24m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.7 MB ? eta -:--:--Downloading torchvision-0.17.2-cp310-cp310-macosx_10_13_x86_64.whl (1.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 M

  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /private/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/pip-req-build-7kb1cc7z


  Resolved https://github.com/facebookresearch/detectron2.git to commit a9c0821a12ad353fb2a96f019515990d5460c5ac
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): finished with status 'done'


  Using cached yacs-0.1.8-py3-none-any.whl.metadata (639 bytes)
  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
  Using cached yacs-0.1.8-py3-none-any.whl.metadata (639 bytes)
  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached fvcore-0.1.5.post20221221-py3-none-any.whl
  Using cached fvcore-0.1.5.post20221221-py3-none-any.whl
  Using cached iopath-0.1.9-py3-none-any.whl.metadata (370 bytes)
  Using cached omegaconf-2.3.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached hydra_core-1.3.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached iopath-0.1.9-py3-none-any.whl.metadata (370 bytes)
  Using cached omegaconf-2.3.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached hydra_core-1.3.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached antlr4_py

  DEPRECATION: Building 'detectron2' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'detectron2'. Discussion can be found at https://github.com/pypa/pip/issues/6334


  Created wheel for detectron2: filename=detectron2-0.6-cp310-cp310-macosx_10_15_x86_64.whl size=870145 sha256=ff2b585654ca20ae0c91baef7f22e3a9bca15dd5c687c62e95854f7f587440e5
  Stored in directory: /private/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/pip-ephem-wheel-cache-prdwp9ut/wheels/47/e5/15/94c80df2ba85500c5d76599cc307c0a7079d0e221bb6fc4375
Successfully built detectron2
  Created wheel for detectron2: filename=detectron2-0.6-cp310-cp310-macosx_10_15_x86_64.whl size=870145 sha256=ff2b585654ca20ae0c91baef7f22e3a9bca15dd5c687c62e95854f7f587440e5
  Stored in directory: /private/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/pip-ephem-wheel-cache-prdwp9ut/wheels/47/e5/15/94c80df2ba85500c5d76599cc307c0a7079d0e221bb6fc4375
Successfully built detectron2
   ━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/23 [werkzeug]Installing collected packages: antlr4-python3-runtime, yacs, werkzeug, tomli, termcolor, tensorboard-data-server, tabulate, pytokens, protobuf, portalocker, pathspec, omega

In [3]:
# Core detection utilities
import os
import warnings
import rasterio
import geopandas as gpd

from detectree2.preprocessing.tiling import tile_data
from detectree2.models.train import setup_cfg
from detectree2.models.predict import predict_on_data
from detectree2.models.outputs import project_to_geojson, stitch_crowns, clean_crowns
from detectron2.engine import DefaultPredictor

def tile_orthomosaic(img_path, tiles_path, buffer=20, tile_width=45, tile_height=45, dtype_bool=True):
    os.makedirs(tiles_path, exist_ok=True)
    tile_data(img_path, tiles_path, buffer, tile_width, tile_height, dtype_bool=dtype_bool)

def setup_detection_config(trained_model_path, device='cpu'):
    cfg = setup_cfg(update_model=trained_model_path)
    try:
        cfg.MODEL.DEVICE = device
    except Exception:
        pass
    return cfg

def predict_tree_crowns(tiles_path, cfg):
    predictor = DefaultPredictor(cfg)
    predict_on_data(directory=tiles_path, predictor=predictor)

def reproject_predictions(tiles_path):
    predictions_folder = os.path.join(tiles_path, 'predictions/')
    geo_predictions_folder = os.path.join(tiles_path, 'predictions_geo/')
    project_to_geojson(tiles_path, predictions_folder, geo_predictions_folder)
    return geo_predictions_folder

def tree_crown_detection_pipeline(img_path, tiles_path, trained_model_path,
                                   buffer=20, tile_width=45, tile_height=45, dtype_bool=True,
                                   device='cpu'):
    # Tile
    tile_orthomosaic(img_path, tiles_path, buffer, tile_width, tile_height, dtype_bool)
    # Config
    cfg = setup_detection_config(trained_model_path, device=device)
    # Predict
    predict_tree_crowns(tiles_path, cfg)
    # Reproject
    geo_predictions_folder = reproject_predictions(tiles_path)
    return geo_predictions_folder

/Users/hbot07/anaconda3/envs/detectree/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/hbot07/anaconda3/envs/detectree/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/hbot07/anaconda3/envs/detectree/lib/python3.10/runpy.py", line 86, in _run_code
    exe

In [5]:
# Step 1: Run detection for all orthomosaics
import os
import glob
import re

project_root = '/Users/hbot07/VS Code/Drone-Phenology-Monitoring'
input_dir = os.path.join(project_root, 'input')
om_dir = os.path.join(input_dir, 'input_om')
tiles_dir = os.path.join(input_dir, 'tiles')
model_path = os.path.join(input_dir, 'detectree_models', '250312_flexi.pth')

# Collect orthomosaics
om_paths = sorted(glob.glob(os.path.join(om_dir, '*.tif')))
if not om_paths:
    raise FileNotFoundError(f"No orthomosaics found in {om_dir}")

print(f"Found {len(om_paths)} orthomosaics. Running detection...")

# Run detection pipeline (inference only)
params = dict(buffer=20, tile_width=45, tile_height=45, dtype_bool=True, device='cpu')

for om_path in om_paths:
    stem = os.path.splitext(os.path.basename(om_path))[0]
    tiles_path = os.path.join(tiles_dir, stem)
    geo_predictions_folder = os.path.join(tiles_path, 'predictions_geo')
    
    print(f"\n=== Processing {stem} ===")
    print("Orthomosaic:", om_path)
    print("Tiles dir  :", tiles_path)
    
    # Skip if predictions already exist
    if os.path.isdir(geo_predictions_folder) and glob.glob(os.path.join(geo_predictions_folder, '*.geojson')):
        print(f"Predictions already exist, skipping detection: {geo_predictions_folder}")
        continue
    
    # Run detection (returns path to predictions_geo/)
    geo_predictions_folder = tree_crown_detection_pipeline(
        img_path=om_path,
        tiles_path=tiles_path,
        trained_model_path=model_path,
        **params
    )
    print(f"Predictions saved to: {geo_predictions_folder}")

print("\nDetection complete for all orthomosaics.")

Found 5 orthomosaics. Running detection...

=== Processing sit_om1 ===
Orthomosaic: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/input/input_om/sit_om1.tif
Tiles dir  : /Users/hbot07/VS Code/Drone-Phenology-Monitoring/input/tiles/sit_om1
Predictions already exist, skipping detection: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/input/tiles/sit_om1/predictions_geo

=== Processing sit_om2 ===
Orthomosaic: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/input/input_om/sit_om2.tif
Tiles dir  : /Users/hbot07/VS Code/Drone-Phenology-Monitoring/input/tiles/sit_om2
Predictions already exist, skipping detection: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/input/tiles/sit_om2/predictions_geo

=== Processing sit_om3 ===
Orthomosaic: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/input/input_om/sit_om3.tif
Tiles dir  : /Users/hbot07/VS Code/Drone-Phenology-Monitoring/input/tiles/sit_om3
Predictions already exist, skipping detection: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/

In [6]:
# Step 2: Confidence sweep and adaptive threshold selection
import numpy as np
import pandas as pd
from detectree2.models.outputs import stitch_crowns, clean_crowns

# Fix PROJ data directory issue
import pyproj
try:
    pyproj_datadir = pyproj.datadir.get_data_dir()
except Exception:
    import subprocess
    result = subprocess.run(['conda', 'info', '--base'], capture_output=True, text=True)
    conda_base = result.stdout.strip()
    proj_data = os.path.join(conda_base, 'share', 'proj')
    if os.path.exists(proj_data):
        pyproj.datadir.set_data_dir(proj_data)
        print(f"Set PROJ_DATA to: {proj_data}")
    else:
        print("Warning: PROJ data directory not found, proceeding anyway...")

# Fixed parameters
fixed_iou = 0.7
fixed_simplify = 0.3
conf_list = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

# Output directory
output_crowns_dir = os.path.join(project_root, 'output', 'input_crowns')
os.makedirs(output_crowns_dir, exist_ok=True)

# Naming function: sit_om{n}.tif -> OM{n}.gpkg
om_num_pat = re.compile(r"sit_om(\d+)", re.IGNORECASE)

def out_name_for(stem: str) -> str:
    m = om_num_pat.search(stem)
    if m:
        return f"OM{int(m.group(1))}.gpkg"
    return f"{stem}_crowns.gpkg"

def load_stitched_for(stem: str):
    geo_folder = os.path.join(tiles_dir, stem, 'predictions_geo')
    if not (os.path.isdir(geo_folder) and glob.glob(os.path.join(geo_folder, '*.geojson'))):
        return None
    gdf = stitch_crowns(geo_folder, 1)
    gdf = gdf[gdf.is_valid]
    return gdf

def compute_count_for_conf(base: gpd.GeoDataFrame, conf: float) -> int:
    g = base.copy()
    if fixed_simplify and fixed_simplify > 0:
        g = g.set_geometry(g.geometry.simplify(fixed_simplify))
    g = clean_crowns(g, fixed_iou, conf)
    g = g[g.is_valid]
    return int(len(g))

# Collect orthomosaic stems
stems = [os.path.splitext(os.path.basename(p))[0] for p in om_paths]

print("Running confidence sweep...")
rows = []
bases = {}

for stem in stems:
    base = load_stitched_for(stem)
    if base is None:
        print(f"Skipping {stem}: predictions not found")
        continue
    bases[stem] = base
    for conf in conf_list:
        count = compute_count_for_conf(base, conf)
        rows.append({'orthomosaic': stem, 'confidence': conf, 'count': count})

if not rows:
    raise RuntimeError('No stitched predictions found. Run detection first.')

sweep_df = pd.DataFrame(rows)
target_count = int(np.median(sweep_df['count']))
print(f"Global median count across all OM×conf: {target_count}")

# Select confidence per OM closest to target
def choose_conf_for_counts(df: pd.DataFrame, target: int) -> pd.Series:
    diffs = (df['count'] - target).abs()
    min_diff = diffs.min()
    candidates = df.loc[diffs == min_diff]
    chosen = candidates.sort_values('confidence', ascending=False).iloc[0]
    return pd.Series({'chosen_conf': float(chosen['confidence']), 'chosen_count': int(chosen['count'])})

chosen = (sweep_df.groupby('orthomosaic')
          .apply(lambda d: choose_conf_for_counts(d.sort_values('confidence'), target_count))
          .reset_index())

print("\nChosen thresholds per orthomosaic:")
display(chosen)

# Save chosen crowns
print("\nSaving crowns with chosen thresholds...")
export_rows = []

for _, r in chosen.iterrows():
    stem = r['orthomosaic']
    conf = float(r['chosen_conf'])
    base = bases.get(stem)
    if base is None:
        continue
    
    # Apply chosen threshold
    g = base.copy()
    if fixed_simplify and fixed_simplify > 0:
        g = g.set_geometry(g.geometry.simplify(fixed_simplify))
    g = clean_crowns(g, fixed_iou, conf)
    g = g[g.is_valid]
    
    # Save
    out_file = os.path.join(output_crowns_dir, out_name_for(stem))
    try:
        g.to_file(out_file, driver='GPKG')
        export_rows.append({
            'orthomosaic': stem, 
            'confidence': conf, 
            'count': int(len(g)), 
            'output': out_file
        })
        print(f"  {stem}: conf={conf:.2f}, count={len(g)} → {out_file}")
    except Exception as e:
        print(f"  Failed to save {stem}: {e}")

print(f"\nAll crowns saved to {output_crowns_dir}")

# Summary
summary = pd.DataFrame(export_rows)
display(summary)

Running confidence sweep...
clearn_crowns: Performing spatial join...
clearn_crowns: Performing spatial join...


clean_crowns: Processing candidate pairs: 100%|██████████| 498/498 [00:00<00:00, 4055.41it/s]



clearn_crowns: Performing spatial join...


clean_crowns: Processing candidate pairs: 100%|██████████| 350/350 [00:00<00:00, 5887.59it/s]



clearn_crowns: Performing spatial join...


...

  sit_om4: conf=0.40, count=80 → /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/input_crowns/OM4.gpkg


clearn_crowns: Performing spatial join...


clean_crowns: Processing candidate pairs: 100%|██████████| 178/178 [00:00<00:00, 5299.18it/s]

  sit_om5: conf=0.50, count=83 → /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/input_crowns/OM5.gpkg

All crowns saved to /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/input_crowns


,orthomosaic,confidence,count,output
0,sit_om1,0.55,82,/Users/hbot07/VS Code/Drone-Phenology-Monitori...
1,sit_om2,0.50,85,/Users/hbot07/VS Code/Drone-Phenology-Monitori...
2,sit_om3,0.45,78,/Users/hbot07/VS Code/Drone-Phenology-Monitori...
3,sit_om4,0.40,80,/Users/hbot07/VS Code/Drone-Phenology-Monitori...
4,sit_om5,0.50,83,/Users/hbot07/VS Code/Drone-Phenology-Monitori...
